In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%PandasProfile
### cell 10 ###

# Compute per-play los_diff based on posTeam using a single groupby
los_diff = (
    df.groupby(['gameId', 'playId', 'nflId'], as_index=False)
      .agg(
          posTeam=('posTeam', 'first'),
          los_max=('los_diff', 'max'),
          los_min=('los_diff', 'min')
      )
      .assign(los_diff=lambda x: x['los_max'].where(x['posTeam'] == 1, x['los_min']))
      .groupby('nflId', as_index=False)
      .agg(los_diff=('los_diff', 'mean'))
)

# Merge back and rename
snap_speed = (
    snap_speed
      .merge(los_diff, on='nflId')
      .rename(columns={'s': 'snap_s', 'a': 'snap_a', 'los_diff': 'snap_los_diff'})
)